## Processamento e Filtragem
**Objetivo**: processamento, filtragem e mapeamento geoespacial de dados cadastrais de estabelecimentos comerciais na cidade de Belém. O foco do fluxo é identificar locais potencialmente relacionados a fontes de ruído urbano (como bares, casas de show e depósitos de bebidas).


* **Carga e Diagnóstico Inicial**: Leitura da base de dados bruta (.csv) de Belém e aplicação de uma função diagnóstica para entender a estrutura e preenchimento inicial das colunas.

### Filtragem por Expressões Regulares (Regex):

* **Inclusão** : Seleção de estabelecimentos contendo palavras-chave específicas (ex: bar, boteco, depósito de cerveja, sinuca, restaurante, casa de show).

* **Exclusão**: Remoção de registros irrelevantes capturados pelo termo genérico "depósito" (ex: depósito de gás, material de construção, água, móveis).


* **Visualização Espacial**: Criação de um mapa interativo dinâmico centralizado nas coordenadas médias dos dados filtrados, plotando cada estabelecimento como um marcador vermelho com balões informativos (popups).

* **Feature Engineering**: Criação de uma nova coluna de endereço consolidado (END_COMPLETO) a partir da junção de múltiplos campos textuais de logradouro.

* **Exportação**: Seleção das colunas de interesse (identificadores, geolocalização e endereço) e salvamento do resultado tratado em uma base de dados limpa (.csv) na pasta de dados processados.

In [1]:
import pandas as pd
from unidecode import unidecode
import folium
from setup_notebook import setup_path
setup_path()
from src.utils.functions import *
from folium.plugins import FastMarkerCluster
import webbrowser
import os
import time
import geopandas as gpd
from src.utils.update_bairros import atualizar_bairros

In [2]:
# 1) LEITURA DO ARQUIVO 
arquivo_csv = '/home/akel/PycharmProjects/city_noise/data/raw/1501402_BELEM.csv'

df = pd.read_csv(arquivo_csv, sep=';', encoding='utf-8', low_memory=False)

inital_describe(df,True)

📊 ANÁLISE EXPLORATÓRIA DO DATAFRAME

📈 DIMENSÕES DO DATASET:
   • 618075 linhas
   • 34 colunas
   • Total de células: 21014550

🔧 TIPOS DE DADOS:
   • str: 16 colunas
   • int64: 11 colunas
   • float64: 7 colunas

🔍 VERIFICAÇÃO DE QUALIDADE DOS DADOS

📝 REGISTROS DUPLICADOS:
   • Total: 0
   • Percentual: 0.00%

❌ VALORES NULOS:
   • Total: 8924268
   • Percentual: 42.47%

📊 COLUNAS COM VALORES NULOS:
   • NOM_TITULO_SEGLOGR: 497814 nulos (80.54%)
   • DSC_MODIFICADOR: 483537 nulos (78.23%)
   • NOM_COMP_ELEM1: 353641 nulos (57.22%)
   • VAL_COMP_ELEM1: 422797 nulos (68.41%)
   • NOM_COMP_ELEM2: 547699 nulos (88.61%)
   • VAL_COMP_ELEM2: 556107 nulos (89.97%)
   • NOM_COMP_ELEM3: 606262 nulos (98.09%)
   • VAL_COMP_ELEM3: 606813 nulos (98.18%)
   • NOM_COMP_ELEM4: 615422 nulos (99.57%)
   • VAL_COMP_ELEM4: 615450 nulos (99.58%)
   • NOM_COMP_ELEM5: 618060 nulos (100.00%)
   • VAL_COMP_ELEM5: 618060 nulos (100.00%)
   • DSC_ESTABELECIMENTO: 545732 nulos (88.30%)
   • COD_INDICADOR_EST

In [3]:
# 2) PRE-FILTRAGEM
# DSC_LOCALIDADE -> BAIRROS
df = df.rename(columns={'DSC_LOCALIDADE': 'BAIRRO'})

#df["BAIRRO"] = df["BAIRRO"].str.upper().apply(unidecode) #maiuculo + remoção de acentuação

df['BAIRRO'] = (
    df['BAIRRO']
    .astype(str)                                # Garante que tudo é string
    .str.upper()                                # Transforma em MAIÚSCULO
    .str.normalize('NFKD')                      # Separa os acentos das letras
    .str.encode('ascii', errors='ignore')       # Remove os acentos isolados
    .str.decode('utf-8')                        # Transforma de volta para texto
    .str.strip()                                # Remove espaços inúteis no início/fim
)

# BAIRRO Belém não existe, portanto:  'BELEM'-> 'INDEFINIDO'
df['BAIRRO'] = df['BAIRRO'].str.replace('BELEM', 'INDEFINIDO')


# 3) FILTRAGEM
# Garante que as colunas estão em maiúsculo e sem NaNs para o filtro funcionar
df['DSC_ESTABELECIMENTO'] = df['DSC_ESTABELECIMENTO'].astype(str).str.upper()

# REGEX
# a. BAR\Boteco isolados
padrao_bar = r'\bBAR\b|\bBOTECO\b'

# b. DEPOSITO DE BEBIDAS: Captura "DEPOSITO DE BEBIDAS", "DEPOSITO DE CERVEJA" ou apenas "DEPOSITO" puro
padrao_deposito = r'DEPOSITO.*BEBIDA|DEPOSITO.*CERVEJA'

# c. Outros padrões
padrao_outros = r'CHURRASC|RESTAURANTE|CASA DE SHOW|CASA DE EVENTOS|RECEPÇÕE|CAFE'
padrao_sinuca = r'\bSINUCA\b|\bBILHAR\b'

# d. Agregando Padrões de Busca
procura_geral = f"{padrao_bar}|{padrao_deposito}|{padrao_sinuca}|{padrao_outros}"


# e. Exclusões explícitas para limpar o "DEPOSITO" genérico de falsos positivos
padrao_exclusao = r'GAS|AGUA|RECICLA|CONSTRUCA|MATERIAL|MOVEIS|FERRO|CIMENTO|' \
                  r'MERCADORIAS|MADEIRA|MATERIAIS|FARINHA|FERRAGENS|MAQUINAS|' \
                  r'CALCADOS|VAGO|TINTAS|MARISCOS'

# FILTROS
# filtro 1: Padrão de busca
df_filt = df[df['DSC_ESTABELECIMENTO'].str.contains(procura_geral, na=False, regex=True)].copy()

# filtro 2: remove termos dentro dos padrões encontrados
df_filt = df_filt[~df_filt['DSC_ESTABELECIMENTO'].str.contains(padrao_exclusao, na=False, regex=True)]

#resetando index
df_filt = df_filt.reset_index(drop=True)
print('============================================================')
print("Amostra dos estabelecimentos encontrados:")
print(df_filt['DSC_ESTABELECIMENTO'].value_counts().head(20))
print('============================================================')
print(df_filt.columns)

Amostra dos estabelecimentos encontrados:
DSC_ESTABELECIMENTO
BAR                          673
RESTAURANTE                  377
DEPOSITO DE BEBIDAS          198
BAR E RESTAURANTE             53
VENDA DE CHURRASCO            22
DEPOSITO DE BEBIDA            21
BAR E MERCEARIA               20
CASA DE EVENTOS               20
CHURRASCARIA                  17
RESTAURANTE SABOR CASEIRO     17
BAR DOS AMIGOS                14
BOTECO                        13
CYBER CAFE                    12
BAR E LANCHONETE              12
RESTAURANTE E LANCHONETE      11
MERCEARIA E BAR               10
CHURRASCO                     10
DEPOSITO BEBIDAS               8
RESTAURANTE BOM SABOR          7
CAFE DA MANHA                  7
Name: count, dtype: int64
Index(['COD_UNICO_ENDERECO', 'COD_UF', 'COD_MUNICIPIO', 'COD_DISTRITO',
       'COD_SUBDISTRITO', 'COD_SETOR', 'NUM_QUADRA', 'NUM_FACE', 'CEP',
       'BAIRRO', 'NOM_TIPO_SEGLOGR', 'NOM_TITULO_SEGLOGR', 'NOM_SEGLOGR',
       'NUM_ENDERECO', 'DSC_MODIFI

In [4]:
# 4) FEATURE ENGINEERING + REMOÇÃO DUPLICATAS

# AGREGAR COLUNAS 
df_filt['END_COMPLETO'] = (df_filt['NOM_TIPO_SEGLOGR'].fillna('') + ' '+df_filt['NOM_TITULO_SEGLOGR'].fillna('') + ' ' + df_filt['NOM_SEGLOGR'].fillna('')).str.strip()
# Novo vetor
df_filt2=df_filt[['COD_DISTRITO','CEP','BAIRRO', 'END_COMPLETO','NUM_ENDERECO','LATITUDE', 'LONGITUDE','DSC_ESTABELECIMENTO', 'COD_INDICADOR_ESTAB_ENDERECO']].copy()

# REMOÇÃO DUPLICATAS
df_filt2 = df_filt2.drop_duplicates(keep='first')
#resetando index
df_filt2 = df_filt2.reset_index(drop=True)

display(df_filt2[df_filt2['BAIRRO']=='Guamá'])
inital_describe(df_filt2,True)

# 5) FUNÇÃ0: ATUALIZA BAIRROS 'INDEFINIDO' A PARTIR DA LOC GEOGRAFICA+ SHAPEFILE DOS BAIRROS
df_filt2 = atualizar_bairros(df_filt2,coluna_bairro='BAIRRO')


,COD_DISTRITO,CEP,BAIRRO,END_COMPLETO,NUM_ENDERECO,LATITUDE,LONGITUDE,DSC_ESTABELECIMENTO,COD_INDICADOR_ESTAB_ENDERECO


📊 ANÁLISE EXPLORATÓRIA DO DATAFRAME

📈 DIMENSÕES DO DATASET:
   • 3981 linhas
   • 9 colunas
   • Total de células: 35829

🔧 TIPOS DE DADOS:
   • int64: 3 colunas
   • str: 3 colunas
   • float64: 3 colunas

🔍 VERIFICAÇÃO DE QUALIDADE DOS DADOS

📝 REGISTROS DUPLICADOS:
   • Total: 0
   • Percentual: 0.00%

❌ VALORES NULOS:
   • Total: 1
   • Percentual: 0.00%

📊 COLUNAS COM VALORES NULOS:
   • COD_INDICADOR_ESTAB_ENDERECO: 1 nulos (0.03%)

✅ ANÁLISE CONCLUÍDA
42 bairros indefinidos processados.
42 bairros atualizados.
0 permaneceram sem bairro.


In [5]:
#6) VISUALIZAÇÃO NO MAPA
centro_lat = df_filt2['LATITUDE'].dropna().mean()
centro_lon = df_filt2['LONGITUDE'].dropna().mean()
centro = [centro_lat, centro_lon]

mapa = folium.Map(location=centro, zoom_start=12)

for _, row in df_filt2.dropna(subset=['LATITUDE', 'LONGITUDE']).iterrows():
    folium.CircleMarker(
        location=[row["LATITUDE"], row["LONGITUDE"]],
        radius=4,
        color="red",
        fill=True,
        fill_color="red",
        # Opcional: adiciona o nome do estabelecimento como popup ao clicar
        popup=row["DSC_ESTABELECIMENTO"] 
    ).add_to(mapa)

# Mostrar mapa
mapa

In [6]:
#7) VISUALIZAÇÃO NO MAPA

df_export = df_filt2.copy()
df_export.to_csv('/home/akel/PycharmProjects/city_noise/data/processed/Bares_etc_Belem_filt.csv', index=False, encoding='utf-8-sig')
print(f"Arquivo salvo com {len(df_export)} registros!")
print("\n#Arquivos salvos", time.strftime("%H:%M:%S"))

Arquivo salvo com 3981 registros!

#Arquivos salvos 14:01:19
